GENERAL FUNCTIONS FOR GF

In [15]:
import numpy as np



def poly_mult_gf2(p1, p2):
    """Multiplies two polynomials in GF(2) using Shift and XOR."""
    # very heavy but with no constraint, we will use it only to calculate g at the beginning
    result = 0
    while p2 > 0:
        if p2 & 1:  
            result ^= p1  # XOR
        p1 <<= 1      # (shift left)
        p2 >>= 1      # (shift right)
        
    return result


def poly_div_gf2(dividend, divisor):
    """Performs polynomial division in GF(2) and returns the remainder."""
    divident_i = dividend
    divisor_len = divisor.bit_length()

    while dividend.bit_length() >= divisor_len:
        shift_amount = dividend.bit_length() - divisor_len
        aligned_divisor = divisor << shift_amount
        dividend ^= aligned_divisor

    if dividend.bit_length() >= divisor_len:
        raise ValueError("Unexpected error: Dividend should be smaller than divisor at this point.")
    else:
        return dividend


# if __name__ == "__main__":
#     msg = 0b101101000   # Messaggio: 45
#     divisor = 0b1101   # Divisore (Generatore): 13
#     parity = poly_div_gf2(msg, divisor)
#     print(f"Dividend: {bin(msg)}")
#     print(f"Divisor:              {bin(divisor)}")
#     print(f"Rest:        {bin(parity)[2:].zfill(divisor.bit_length() - 1)}")

def generate_gf_tables(m, prim_poly=None):
    """
    Genera exp_table e log_table per il campo di Galois GF(2^m).
    
    :param m: Esponente del campo (es. 13 per GF(2^13))
    :param prim_poly: (Opzionale) Polinomio primitivo in formato intero. 
                      Se None, cerca nel dizionario interno.
    :return: (exp_table, log_table) come liste di interi.
    """
    
    # Dizionario dei polinomi primitivi standard (includono il bit m-esimo)
    standard_polys = {
        3: 0b1011,     # x^3 + x + 1
        4: 0x13,       # x^4 + x + 1 (usato spesso negli esempi didattici)
        8: 0x11D,      # x^8 + x^4 + x^3 + x^2 + 1 (Standard Reed-Solomon/AES)
        13: 0x201B,    # x^13 + x^4 + x^3 + x + 1 (Il tuo campo per BCH)
        16: 0x1100B    # x^16 + x^12 + x^3 + x + 1
    }
    
    if prim_poly is None:
        if m not in standard_polys:
            raise ValueError(f"Polinomio primitivo per m={m} non predefinito. Inseriscilo manualmente.")
        prim_poly = standard_polys[m]
        
    field_size = 1 << m  # 2^m (es. 8192 per m=13)
    
    # Inizializziamo le liste con zeri
    exp_table = [0] * field_size
    log_table = [0] * field_size
    
    # x rappresenta il valore corrente di alpha^i
    x = 1 
    
    # Iteriamo per tutti gli elementi non nulli del campo (da 0 a 2^m - 2)
    for i in range(field_size - 1):
        exp_table[i] = x
        log_table[x] = i
        
        # Moltiplicare per alpha equivale a fare uno shift a sinistra di 1 bit
        x <<= 1 
        
        # Se lo shift ha acceso il bit m-esimo (overflow rispetto al campo),
        # dobbiamo fare la riduzione modulo il polinomio primitivo (tramite XOR)
        if x & field_size: 
            x ^= prim_poly
            
    # L'elemento alpha^(2^m - 1) è uguale ad alpha^0, ovvero 1. 
    # Chiudiamo il ciclo.
    exp_table[field_size - 1] = 1 
    
    # Nota: log_table[0] matematicamente non è definito. 
    # Lo lasciamo a 0 per comodità, ma non andrebbe mai interrogato.
    
    return exp_table, log_table

exp_table, log_table = generate_gf_tables(m=4)


def gf_mult(a, b):
    if a == 0 or b == 0:
        return 0
    mod_value = len(exp_table) - 1
    index = (log_table[a] + log_table[b]) % mod_value # (o usiamo il modulo a facciamo la LUT doppia)
    return exp_table[index]

def gf_inv(a):
    if a == 0:
        raise ZeroDivisionError("0 non ha inverso in GF")
    mod_value = len(exp_table) - 1
    return exp_table[(mod_value - log_table[a]) % mod_value]





ENCODING FUNCTIONS

In [18]:
def get_g_polynomial(m, t=2):
    """
    Finds the generator polynomial g(x) for a BCH code with error correction capability t in GF(2^m).
    """
    
    min_polys = {
        3: {
            1: 0b1011,      # x^3 + x + 1
            3: 0b1101       # x^3 + x^2 + 1
        },
        4: {
            1: 0b10011,     # x^4 + x + 1
            3: 0b11111      # x^4 + x^3 + x^2 + x + 1
        },
        5: {
            1: 0b100101,    # x^5 + x^2 + 1
            3: 0b111101     # x^5 + x^4 + x^3 + x^2 + 1
        },
        13: {
            1: 0x201B,      # x^13 + x^4 + x^3 + x + 1
            3: 0x39E3       # minimal polynomial for alpha^1, alpha^2, alpha^3, alpha^4, alpha^5, alpha^6 (t=2)
        }
    }

    if m not in min_polys:
        raise ValueError(f"Dati per m={m} non disponibili.")

    m1 = min_polys[m][1]
    
    if t == 1:
        return m1
    elif t == 2:
        m3 = min_polys[m][3]
        # Usiamo la tua funzione poly_mult_gf2 che hai già scritto
        return poly_mult_gf2(m1, m3)
    else:
        raise ValueError("Questa funzione è configurata solo per t=1 o t=2.")



def encoding(word, g):
    parity_bits = g.bit_length() - 1
    word_s = word << parity_bits
    remainder = poly_div_gf2(word_s, g)
    codeword = word_s ^ remainder
    return codeword

# if __name__ == "__main__":
#     # Il tuo messaggio di 4 bit
#     word = 0b1001 
    
#     # 1. Creiamo il polinomio generatore per t=2
#     g = get_g_polynomial(m=4, t=2)
#     print(f"Polinomio Generatore g(x): {bin(g)}") # Dovrebbe essere 0b1111111 (grado 6)
    
#     # 2. Codifichiamo la parola
#     codeword = encoding(word, g)
#     print(f"Messaggio originale:       {bin(word)}")
#     print(f"Codeword finale:           {bin(codeword)}")


SYNDROMES RESEARCH

In [19]:
# starting with the syndromes calculations, using the Horner method
# the lenght of the LUT strictly depends on m
# for m = 13 P(x) = x^13 + x^4 + x^3 + x + 1 (1010000000001 in binary)


exp_table, log_table = generate_gf_tables(m=4)
# # EXAMPLE WITH M = 3
# # Ex: alpha^2 = 4. (because  001 -> a^0, 010 -> a^1, 100 -> a^2,  011 -> a^3, 110 -> a^4, 101 -> a^5, 111 -> a^6)
# exp_table = [1, 2, 4, 3, 6, 7, 5,   1, 2, 4, 3, 6, 7, 5] 

# # Ex: alpha^3. log_table[4] = 2.
# # Index 0 is not defined (log of 0 is undefined), so we can set it to 0 or -1 as a placeholder.
# log_table = [0, 0, 1, 3, 2, 6, 4, 5]

def calculate_horner(msg, root_idx):
    root = exp_table[root_idx]
    s_i = 0
    msg_bits = msg.bit_length()
    for bit_pos in range(msg_bits - 1, -1, -1): # most to least significant bit
            # Estraiamo il singolo bit
            bit = (msg >> bit_pos) & 1
            # METODO DI HORNER: Sindrome = (Sindrome_precedente * radice) XOR bit
            s_i = gf_mult(s_i, root) ^ bit
    return s_i

def calculate_syndromes(msg, t):
    """Calculates the syndromes S_1, S_2, ..., S_{2t} for a received message. """
    s1 = calculate_horner(msg, root_idx=1)
    s2 = gf_mult(s1, s1)
    if t == 1:
        return [s1, s2]
    else:
        s3 = calculate_horner(msg, root_idx=3)
        s4 = gf_mult(s2, s2)
        return [s1, s2, s3, s4]


if __name__ == "__main__":
    
    perfect_codeword = 0b1001111011
    corrupted_codeword = 0b1101111011
    print("No errors")
    s_perf = calculate_syndromes(perfect_codeword, t=2)
    print(f"Sindromi: S_1 = {s_perf[0]}, S_2 = {s_perf[1]}, S_3 = {s_perf[2]}, S_4 = {s_perf[3]}")
    print("\nCorrupted")
    s_corr = calculate_syndromes(corrupted_codeword, t=2)
    print(f"Sindromi: S_1 = {s_corr[0]}, S_2 = {s_corr[1]}, S_3 = {s_corr[2]}, S_4 = {s_corr[3]}")


No errors
Sindromi: S_1 = 8, S_2 = 12, S_3 = 10, S_4 = 15

Corrupted
Sindromi: S_1 = 13, S_2 = 14, S_3 = 0, S_4 = 11


DECODING FUNCTIONS -- Berlekamp-Massey

In [20]:


## LOCALIZATOR POLYNOMIAL LAMBDA (using Berlekamp-Massey Algorithm)

def bch_berlekamp_massey(syndromes, t=2):
    """
    syndromes: [S1, S2, S3, S4]
    Returns lambda coefficients
    """
    lambda_poly = [1]        # starting point
    b_poly = [1]             # support polynomial
    L = 0                    # current degree of lambda

    for k in range(1, (2*t) + 1):
        # d: discrepancy
        # d = S_k + sum(Lambda_i * S_{k-i})
        d = syndromes[k-1]
        for i in range(1, L + 1):
            if i < len(lambda_poly): # se è più corto significa che non considera degli zeri in coda
                term = gf_mult(lambda_poly[i], syndromes[k-1-i]) # remember that k-1-i + i = k-1
                d ^= term
        # Shift B(x) = B(x) * x
        b_poly.insert(0, 0) # B(x) must grow with time
        
        if d != 0:
            T = list(lambda_poly)
            
            # check for length and pad with zeros if necessary
            max_len = max(len(lambda_poly), len(b_poly)) 
            lambda_poly += [0] * (max_len - len(lambda_poly)) # pad with zeros if necessary
            
            # 2. update: Lambda(x) = Lambda(x) + d * B(x)
            for i in range(len(b_poly)):
                lambda_poly[i] ^= gf_mult(d, b_poly[i])
            
            if 2 * L < k:
                # 3. update B(x) = Lambda(x) / d
                L = k - L
                d_inv = gf_inv(d)
                b_poly = [gf_mult(val, d_inv) for val in T]
                
    # Pulisce gli zeri in coda e limita a t+1
    lambda_poly += [0] * ((t + 1) - len(lambda_poly))
    return lambda_poly[:t+1]



CHIEN SEARCH

In [21]:
def bch_chien_search(lambda_poly, message_length=4680, m=13):
    """
    lambda_poly: [1, Lambda_1, Lambda_2] (t=2)
    RETRURN: list of error positions (indices)
    """
    error_positions = []
    
    alpha_inv_1 = exp_table[2**m - 2] 
    alpha_inv_2 = exp_table[2**m - 3] 
    
    # Inizializziamo i termini con i coefficienti del polinomio per il bit 0
    term1 = lambda_poly[1]
    term2 = lambda_poly[2]
    
    # Scansioniamo la word bit per bit
    for i in range(message_length):
        
        # 1. Valutazione: Il polinomio si annulla in questa posizione?
        # Stiamo calcolando: 1 ^ term1 ^ term2
        if 1 ^ term1 ^ term2 == 0:
            error_positions.append(i)
            # Se siamo su un microcontrollore
            # message_array[i / 8] ^= (1 << (i % 8)); // Flip del bit!
            
        # 2. Aggiornamento di Chien per il prossimo giro di loop
        term1 = gf_mult(term1, alpha_inv_1)
        term2 = gf_mult(term2, alpha_inv_2)
        
    return error_positions



PROVA

In [30]:
if __name__ == "__main__":
    
    m = 4
    word = 0b1001
    codeword = encoding(word, get_g_polynomial(m))
    print(f"Codeword: {bin(codeword)}")
    corrupted_codeword = codeword ^ 0b10010000 # introduce an error
    s_corr = calculate_syndromes(corrupted_codeword, t=2)
    lambda_coeffs = bch_berlekamp_massey(s_corr, t=2)
    error_positions = bch_chien_search(lambda_coeffs, message_length=10, m=4)
    print(f"Error positions: {error_positions}")



Codeword: 0b100111001100
Error positions: [4, 7]


In [ ]:
def generate_gf13_header(filename="gf13_luts.h"):
    m = 13
    size = (1 << m) - 1  # 8191
    
    # Primitive Polynomial: x^13 + x^4 + x^3 + x + 1
    # 0x201B represents the polynomial including the x^13 bit
    poly = 0x201B 
    
    exp_table = [0] * (size * 2)
    log_table = [0] * (size + 1)
    
    current = 1
    for i in range(size):
        exp_table[i] = current
        log_table[current] = i
        
        # Step to next power: alpha^(i+1)
        current <<= 1
        if current & (1 << m): # If overflow bit 13 is set
            current ^= poly
            
    # Duplicate the exp_table for the "no-modulo" multiplication trick
    for i in range(size):
        exp_table[i + size] = exp_table[i]
        
    # Write to C++ Header File
    with open(filename, "w") as f:
        f.write("#ifndef GF13_LUTS_H\n#define GF13_LUTS_H\n\n")
        f.write("#include <stdint.h>\n\n")
        
        # Write Exp Table
        f.write(f"const uint16_t exp_table[{len(exp_table)}] = {{\n    ")
        for i, val in enumerate(exp_table):
            f.write(f"{val},")
            if (i + 1) % 12 == 0: f.write("\n    ")
        f.write("\n};\n\n")
        
        # Write Log Table
        f.write(f"const uint16_t log_table[{len(log_table)}] = {{\n    ")
        for i, val in enumerate(log_table):
            f.write(f"{val},")
            if (i + 1) % 12 == 0: f.write("\n    ")
        f.write("\n};\n\n")
        
        f.write("#endif // GF13_LUTS_H\n")
    
    print(f"Header file '{filename}' generated successfully.")

# if __name__ == "__main__":
#     generate_gf13_header()